In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

In [ ]:
TASK = Path.cwd().parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
CONFIG = json.loads((TASK.parent/'config.json').read_text())
Y = CONFIG['last_year']
REFERENCE = 'empirical_complete'
MODELS = CONFIG['primary_plot_models']
DRAWS = 2000
PERCENTILES = np.linspace(.01,.99,99)
NORMAL = stats.norm.ppf(PERCENTILES)
rng = np.random.default_rng(CONFIG['seed'])

In [ ]:
panel = pd.read_csv(INPUT/'fit/data/panel.csv',index_col='dblp_id').to_numpy(dtype=float).T
sources = {REFERENCE:panel[:,np.isfinite(panel).all(axis=0)]}
sources.update({source:np.load(INPUT/source/'trajectories.npy') for source in MODELS})
totals = {source:array[Y-3:Y+1].sum(axis=0) for source,array in sources.items()}
positive = {source:np.log(values[values>0]) for source,values in totals.items()}
n_careers = min(map(len,totals.values()))
n_positive = min(map(len,positive.values()))

In [ ]:
def qq_summary(log_totals):
    standardized = (log_totals-log_totals.mean(axis=-1,keepdims=True))/log_totals.std(axis=-1,ddof=1,keepdims=True)
    residual = np.quantile(standardized,PERCENTILES,axis=-1).T - NORMAL
    # same normal plotting positions as the existing QQ R² diagnostic
    theoretical = stats.probplot(log_totals.shape[-1]*[0],fit=False)[0]
    theoretical = theoretical-theoretical.mean()
    ordered = np.sort(standardized,axis=-1)
    r_squared = (ordered @ theoretical)**2/(np.sum(ordered**2,axis=-1)*np.sum(theoretical**2))
    return residual,r_squared

In [ ]:
curves, boot_curves, qq, boot_qq, boot_zero = {}, {}, {}, {}, {}
for source,logs in positive.items():
    curves[source],qq[source] = qq_summary(logs)
    # resample career totals within the positive subset, matched across sources
    boot_curves[source],boot_qq[source] = qq_summary(rng.choice(logs,(DRAWS,n_positive)))
    # separate unconditional career resample for the zero mass
    boot_zero[source] = (rng.choice(totals[source],(DRAWS,n_careers))==0).mean(axis=1)

In [ ]:
curve_rows, rows = [], []
for source,curve in curves.items():
    lower,upper = np.quantile(boot_curves[source],[.025,.975],axis=0)
    curve_rows.append(pd.DataFrame({'source':source, 'percentile':PERCENTILES,
                                    'residual':curve, 'lower':lower, 'upper':upper}))
    zero_lower,zero_upper = np.quantile(boot_zero[source],[.025,.975])
    qq_lower,qq_median,qq_upper = np.quantile(boot_qq[source],[.025,.5,.975])
    rms_gap = np.sqrt(np.mean((curve-curves[REFERENCE])**2))
    rows.append({'source':source, 'n_careers':len(totals[source]), 'n_positive':len(positive[source]),
                 'matched_n_careers':n_careers, 'matched_n_positive':n_positive, 'draws':DRAWS,
                 'zero_fraction':np.mean(totals[source]==0), 'zero_lower':zero_lower, 'zero_upper':zero_upper,
                 'qq_r2':qq[source], 'qq_r2_lower':qq_lower, 'qq_r2_median':qq_median, 'qq_r2_upper':qq_upper,
                 'rms_gap_from_dblp':rms_gap})
summary = pd.DataFrame(rows)
pd.concat(curve_rows,ignore_index=True).to_csv(OUTPUT/'lognormal_residual_curves.csv',index=False)
summary.to_csv(OUTPUT/'lognormal_residual_summary.csv',index=False)
display(summary)

In [ ]:
# pairwise ordering frequencies from independent bootstrap replicates
gaps = {source:np.sqrt(np.mean((boot_curves[source]-boot_curves[REFERENCE])**2,axis=1))
        for source in MODELS}
pairs = [(left,right) for i,left in enumerate(sources) for right in list(sources)[i+1:]]
ordering = pd.DataFrame([{'left':left, 'right':right,
                          'fraction_left_higher_qq_r2':np.mean(boot_qq[left]>boot_qq[right]),
                          'fraction_left_closer_to_dblp':np.mean(gaps[left]<gaps[right])
                          if left in gaps and right in gaps else np.nan}
                         for left,right in pairs])
ordering.to_csv(OUTPUT/'lognormal_residual_ordering.csv',index=False)
display(ordering)